# 04 — Path patching e validação do circuito

Activation patching mede o efeito causal de uma ativação sobre o resultado final, mas mistura todos os caminhos downstream.

Neste notebook vamos:

1. medir o efeito total de heads sender;
2. isolar arestas `sender → receiver` com path patching;
3. testar **faithfulness**, **completeness** e **minimality** do circuito proposto.

Os resultados são específicos para estes prompts, esta versão do modelo e esta intervenção.

In [1]:
from functools import partial

import plotly.graph_objects as go
import torch
from transformer_lens import patching
import transformer_lens.utilities as utils

from ioi_replication import (
    build_ioi_examples,
    load_gpt2_small,
    logits_to_logit_diff,
    normalized_recovery,
    select_device,
    tokenize_answers,
)

device = select_device()
model = load_gpt2_small(device)
prompts, answers = build_ioi_examples()
tokens = model.to_tokens(prompts, prepend_bos=True).to(device)
answer_tokens = tokenize_answers(model, answers, device)
swap_indices = torch.tensor(
    [i + 1 if i % 2 == 0 else i - 1 for i in range(len(tokens))],
    device=device,
)
corrupted_tokens = tokens[swap_indices]

with torch.no_grad():
    clean_logits, clean_cache = model.run_with_cache(tokens)
    corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)

clean_diff = logits_to_logit_diff(clean_logits, answer_tokens)
corrupted_diff = logits_to_logit_diff(corrupted_logits, answer_tokens)


def ioi_metric(logits: torch.Tensor) -> torch.Tensor:
    return normalized_recovery(
        logits_to_logit_diff(logits, answer_tokens),
        clean_diff,
        corrupted_diff,
    )


print(f"Dispositivo: {device}")
print(f"Limpo={clean_diff.item():+.3f} | corrompido={corrupted_diff.item():+.3f}")

C:\Users\Marcos\Documents\IOI Replication\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:  10%|█         | 15/148 [00:00<00:00, 149.98it/s]

Loading weights:  24%|██▎       | 35/148 [00:00<00:00, 179.33it/s]

Loading weights:  36%|███▌      | 53/148 [00:00<00:00, 122.14it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 359.89it/s]

Dispositivo: cuda
Limpo=+3.552 | corrompido=-3.552


In [2]:
CIRCUIT = {
    "name mover": [(9, 9), (10, 0), (9, 6)],
    "backup name mover": [
        (10, 10), (10, 6), (10, 2), (10, 1),
        (11, 2), (9, 7), (9, 0), (11, 9),
    ],
    "negative name mover": [(10, 7), (11, 10)],
    "s2 inhibition": [(7, 3), (7, 9), (8, 6), (8, 10)],
    "induction": [(5, 5), (5, 8), (5, 9), (6, 9)],
    "duplicate token": [(0, 1), (0, 10), (3, 0)],
    "previous token": [(2, 2), (4, 11)],
}

circuit_heads = {head for heads in CIRCUIT.values() for head in heads}
print(f"Grupos: {len(CIRCUIT)} | heads únicas: {len(circuit_heads)}")
for group, heads in CIRCUIT.items():
    print(f"{group:22s}: {', '.join(f'{layer}.{head}' for layer, head in heads)}")

Grupos: 7 | heads únicas: 26
name mover            : 9.9, 10.0, 9.6
backup name mover     : 10.10, 10.6, 10.2, 10.1, 11.2, 9.7, 9.0, 11.9
negative name mover   : 10.7, 11.10
s2 inhibition         : 7.3, 7.9, 8.6, 8.10
induction             : 5.5, 5.8, 5.9, 6.9
duplicate token       : 0.1, 0.10, 3.0
previous token        : 2.2, 4.11


## 1. Sender → resultado final

Primeiro aplicamos patch na saída `z` de uma sender inteira. Isso mede seu efeito causal total através de **todos** os caminhos downstream.

Depois compararemos esse valor com arestas individuais. Uma aresta pequena não contradiz um efeito total grande: a sender pode distribuir informação por vários receivers.

In [3]:
def patch_sender_output(
    corrupted_z: torch.Tensor,
    hook,
    *,
    head: int,
) -> torch.Tensor:
    patched = corrupted_z.clone()
    patched[:, :, head, :] = clean_cache[hook.name][:, :, head, :]
    return patched


upstream_senders = CIRCUIT["duplicate token"] + CIRCUIT["induction"]
s_inhibition_senders = CIRCUIT["s2 inhibition"]
selected_senders = upstream_senders + s_inhibition_senders
sender_to_final = {}

with torch.no_grad():
    for layer, head in selected_senders:
        logits = model.run_with_hooks(
            corrupted_tokens,
            fwd_hooks=[
                (
                    utils.get_act_name("z", layer),
                    partial(patch_sender_output, head=head),
                )
            ],
        )
        sender_to_final[(layer, head)] = ioi_metric(logits).item()

figure = go.Figure(
    data=go.Bar(
        x=[f"{layer}.{head}" for layer, head in selected_senders],
        y=[sender_to_final[sender] for sender in selected_senders],
    )
)
figure.update_layout(
    title="Recuperação total por sender",
    xaxis_title="sender head",
    yaxis_title="recuperação normalizada",
)
figure.show()

## 2. Isolando sender → receiver

No path patching direto, calculamos a escrita `limpa − corrompida` da sender e a adicionamos somente à entrada de uma receiver. Os demais readers continuam vendo a sender corrompida.

Testaremos duas famílias de arestas:

- Duplicate/Induction → S-Inhibition pela entrada **V**;
- S-Inhibition → Name Movers pela entrada **Q**.

Cada célula dos mapas abaixo é uma intervenção em uma única aresta.

In [4]:
s_inhibition_receivers = CIRCUIT["s2 inhibition"]
upstream_to_s_inhibition = torch.zeros(
    len(upstream_senders),
    len(s_inhibition_receivers),
    device=device,
)

with torch.no_grad():
    for sender_index, (source_layer, source_head) in enumerate(upstream_senders):
        all_receivers = patching.get_act_patch_direct_path(
            model=model,
            corrupted_tokens=corrupted_tokens,
            clean_cache=clean_cache,
            corrupted_cache=corrupted_cache,
            patching_metric=ioi_metric,
            src_layer=source_layer,
            src_head=source_head,
            component="v",
            verbose=False,
        )
        for receiver_index, receiver in enumerate(s_inhibition_receivers):
            upstream_to_s_inhibition[sender_index, receiver_index] = all_receivers[receiver]

figure = go.Figure(
    data=go.Heatmap(
        z=upstream_to_s_inhibition.detach().cpu().numpy(),
        x=[f"{layer}.{head}" for layer, head in s_inhibition_receivers],
        y=[f"{layer}.{head}" for layer, head in upstream_senders],
        colorscale="RdBu",
        zmid=0,
        colorbar_title="recuperação",
        hovertemplate="sender=%{y}<br>receiver=%{x}<br>recuperação=%{z:.3f}<extra></extra>",
    )
)
figure.update_layout(
    title="Arestas Duplicate/Induction → S-Inhibition (entrada V)",
    xaxis_title="receiver",
    yaxis_title="sender",
    height=520,
)
figure.show()

In [5]:
name_mover_receivers = CIRCUIT["name mover"]
s_inhibition_to_name_mover = torch.zeros(
    len(s_inhibition_senders),
    len(name_mover_receivers),
    device=device,
)

with torch.no_grad():
    for sender_index, (source_layer, source_head) in enumerate(s_inhibition_senders):
        all_receivers = patching.get_act_patch_direct_path(
            model=model,
            corrupted_tokens=corrupted_tokens,
            clean_cache=clean_cache,
            corrupted_cache=corrupted_cache,
            patching_metric=ioi_metric,
            src_layer=source_layer,
            src_head=source_head,
            component="q",
            verbose=False,
        )
        for receiver_index, receiver in enumerate(name_mover_receivers):
            s_inhibition_to_name_mover[sender_index, receiver_index] = all_receivers[receiver]

figure = go.Figure(
    data=go.Heatmap(
        z=s_inhibition_to_name_mover.detach().cpu().numpy(),
        x=[f"{layer}.{head}" for layer, head in name_mover_receivers],
        y=[f"{layer}.{head}" for layer, head in s_inhibition_senders],
        colorscale="RdBu",
        zmid=0,
        colorbar_title="recuperação",
        hovertemplate="sender=%{y}<br>receiver=%{x}<br>recuperação=%{z:.3f}<extra></extra>",
    )
)
figure.update_layout(
    title="Arestas S-Inhibition → Name Movers (entrada Q)",
    xaxis_title="receiver",
    yaxis_title="sender",
    height=430,
)
figure.show()

## 3. Validação do circuito por mean ablation

Para testar o circuito como conjunto, substituiremos as heads fora dele por ativações médias de prompts-controle ABC, com três nomes distintos.

- **Faithfulness:** o circuito preserva o comportamento do modelo completo?
- **Completeness:** após uma intervenção incomum, o circuito e o modelo completo falham de modo semelhante?
- **Minimality:** remover um grupo incluído prejudica o circuito?

Usaremos a diferença de logits bruta nesta seção, para tornar as comparações explícitas.

In [6]:
control_prompts = [
    "When Alice and Bob went to the shops, Carol gave the bag to",
    "When Carol and Alice went to the shops, Bob gave the bag to",
    "When Alice and Bob went to the park, Carol gave the ball to",
    "When Carol and Alice went to the park, Bob gave the ball to",
    "When Alice and Bob went to the shops, Carol gave an apple to",
    "When Carol and Alice went to the shops, Bob gave an apple to",
    "After Alice and Bob went to the park, Carol gave a drink to",
    "After Carol and Alice went to the park, Bob gave a drink to",
]
control_tokens = model.to_tokens(control_prompts, prepend_bos=True).to(device)
assert control_tokens.shape == tokens.shape

with torch.no_grad():
    _, control_cache = model.run_with_cache(
        control_tokens,
        names_filter=lambda name: name.endswith("attn.hook_z"),
    )

control_means = {
    layer: control_cache["z", layer].mean(dim=0, keepdim=True)
    for layer in range(model.cfg.n_layers)
}
print("Controles ABC tokenizados com shape", tuple(control_tokens.shape))

Controles ABC tokenizados com shape (8, 15)


In [7]:
def mean_ablate_heads(
    z: torch.Tensor,
    hook,
    *,
    layer: int,
    keep_heads: set[tuple[int, int]],
) -> torch.Tensor:
    ablated = z.clone()
    for head in range(model.cfg.n_heads):
        if (layer, head) not in keep_heads:
            ablated[:, :, head, :] = control_means[layer][:, :, head, :]
    return ablated


def run_with_kept_heads(keep_heads: set[tuple[int, int]]) -> torch.Tensor:
    hooks = [
        (
            utils.get_act_name("z", layer),
            partial(mean_ablate_heads, layer=layer, keep_heads=keep_heads),
        )
        for layer in range(model.cfg.n_layers)
    ]
    with torch.no_grad():
        return model.run_with_hooks(tokens, fwd_hooks=hooks)


all_heads = {
    (layer, head)
    for layer in range(model.cfg.n_layers)
    for head in range(model.cfg.n_heads)
}
assert circuit_heads < all_heads

In [8]:
circuit_logits = run_with_kept_heads(circuit_heads)
circuit_diff = logits_to_logit_diff(circuit_logits, answer_tokens)
faithfulness_ratio = circuit_diff / clean_diff

print(f"Modelo completo: {clean_diff.item():+.3f}")
print(f"Somente circuito: {circuit_diff.item():+.3f}")
print(f"Faithfulness: {faithfulness_ratio.item():.1%}")
assert torch.isfinite(faithfulness_ratio)

Modelo completo: +3.552
Somente circuito: +2.882
Faithfulness: 81.1%


In [9]:
main_name_movers = set(CIRCUIT["name mover"])

full_without_main_logits = run_with_kept_heads(all_heads - main_name_movers)
circuit_without_main_logits = run_with_kept_heads(circuit_heads - main_name_movers)

full_without_main = logits_to_logit_diff(full_without_main_logits, answer_tokens)
circuit_without_main = logits_to_logit_diff(circuit_without_main_logits, answer_tokens)
completeness_gap = full_without_main - circuit_without_main

print("Após remover as Name Movers principais:")
print(f"  modelo completo: {full_without_main.item():+.3f}")
print(f"  circuito:        {circuit_without_main.item():+.3f}")
print(f"  gap:             {completeness_gap.item():+.3f}")
print("Um gap positivo sugere comportamento de backup fora do circuito mantido.")

Após remover as Name Movers principais:


  modelo completo: +3.218
  circuito:        +2.392
  gap:             +0.826
Um gap positivo sugere comportamento de backup fora do circuito mantido.


In [10]:
group_removed_diffs = {}
group_importance = {}

for group, group_heads in CIRCUIT.items():
    kept_heads = circuit_heads - set(group_heads)
    removed_logits = run_with_kept_heads(kept_heads)
    removed_diff = logits_to_logit_diff(removed_logits, answer_tokens)
    group_removed_diffs[group] = removed_diff.item()
    group_importance[group] = (circuit_diff - removed_diff).item()

figure = go.Figure(
    data=go.Bar(
        x=list(group_importance),
        y=list(group_importance.values()),
    )
)
figure.update_layout(
    title="Queda no logit diff ao remover cada grupo do circuito",
    xaxis_title="grupo removido",
    yaxis_title="circuito intacto − circuito sem grupo",
)
figure.show()

for group in CIRCUIT:
    print(
        f"{group:22s}: sem grupo={group_removed_diffs[group]:+.3f} | "
        f"queda={group_importance[group]:+.3f}"
    )

name mover            : sem grupo=+2.392 | queda=+0.490
backup name mover     : sem grupo=+2.571 | queda=+0.310
negative name mover   : sem grupo=+5.174 | queda=-2.292
s2 inhibition         : sem grupo=+0.419 | queda=+2.463
induction             : sem grupo=+1.489 | queda=+1.392
duplicate token       : sem grupo=+1.988 | queda=+0.893
previous token        : sem grupo=+1.226 | queda=+1.656


## Conclusão

Os testes respondem perguntas diferentes:

- **Path patching:** por qual receiver a informação de uma sender influencia o resultado?
- **Faithfulness:** o conjunto proposto reproduz boa parte do comportamento normal?
- **Completeness:** o circuito acompanha o modelo em intervenções fora da distribuição?
- **Minimality:** quais grupos fazem diferença quando removidos?

Uma contribuição pequena não prova irrelevância: caminhos paralelos podem compensar a remoção. Uma contribuição negativa também pode ser funcional — por exemplo, Negative Name Movers devem reduzir a diferença de logits correta por construção.

Esta replicação valida a estrutura qualitativa do circuito, mas os números exatos são dependentes da versão, dos prompts e do tipo de ablação.